# Scryfall Tags: Multi-Label Classification  
__Objective:__ Since the scryfall tags are in essence a collection of multiple labels for each card, this problem is at it's core a mutli-label classification task. However, given that there are nearly as many labels as cards, it will be easier to frame this as a seq2seq problem.

## Packages and Data

In [1]:
# UNCOMMENT if operating in google colab

## mount to drive
from google.colab import drive
drive.mount('/content/gdrive', force_remount=True)

## ensure current directory is identified
import os
PROJECT_PATH = '/content/gdrive/MyDrive/data_science/scryfall-llm-sandbox'
os.chdir(f"{PROJECT_PATH}/notebooks")
print(f"Current Directory: {os.getcwd()}")

# check that GPUs are available
import torch
print(f'GPUs Available?: {torch.cuda.is_available()}')
print(f"Device Name: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'No GPU'}")

Mounted at /content/gdrive
Current Directory: /content/gdrive/MyDrive/data_science/scryfall-llm-sandbox/notebooks
GPUs Available?: True
Device Name: Tesla T4


In [2]:
# connect project directory
import sys
from pathlib import Path
dir = str(Path(Path.cwd()).parents[0])
if dir not in sys.path:
    sys.path.append(dir)

## UNCOMMENT if operating in gdrive colab
gdir = str(Path.cwd()) +  '/MyDrive/data_science/scryfall-llm-sandbox'
if gdir not in sys.path:
  sys.path.append(gdir)

In [3]:
# params

## data gathering
from src.config import BUILD_DATASET, TASK, TAG_SIZE, DATASET_SIZE_N, TEST_SIZE_N
from src.config import MAX_INPUT_LENGTH, MAX_TARGET_LENGTH

## modeling
from src.config import MODEL_NAME
from src.config import BATCH_SIZE, LEARNING_RATE, WEIGHT_DECAY, NUM_EPOCHS
from src.config import GENERATION_MAX_LENGTH, GENERATION_NUM_BEAMS

## save model
from src.config import OUTPUT_DIR

In [4]:
# packages

## load from project directory
from src.data_gathering.scryfall_dataset import ScryfallDataset
if TASK == 'seq2seq':
    from src.fine_tuning.fine_tune_seq2seq import FineTuneLLM
elif TASK == 'multi_label_classification':
    from src.fine_tuning.fine_tune_multi_lab import FineTuneLLM

In [5]:
# get data
sf = ScryfallDataset(task = TASK)

## build dataset as needed
if BUILD_DATASET:
    sf.build_dataset(
        tag_path = '../reports/scryfall_tags.json',
        train_size_pct = 0.8,
        truncate_dataset = DATASET_SIZE_N,
        test_size_n = TEST_SIZE_N,
        top_n_tags = TAG_SIZE
    )

## load dataset
sf.load_hf_dataset(
    train_path = f'../data/scryfall_{TASK}_train.json',
    val_path = f'../data/scryfall_{TASK}_val.json',
    test_path = f'../data/scryfall_{TASK}_test.json'
)

Scryfall Tag Question Answering Dataset Built
	Train Records = 6717
	Validation Records = 1680
	Test Records = 50
	Records saved to...
		../data/scryfall_multi_label_classification_train.json
		../data/scryfall_multi_label_classification_val.json
		../data/scryfall_multi_label_classification_test.json
	NOTE: This method does not create the huggingface dataset object. Run load_dataset() for that.
Scryfall Tag Multi Label Classification Dataset Loaded
	Train Records = 6717
	Val Records = 1680
	Test Records = 50
	Count Unique Tags = 300


## Modeling

__distilbert-base-uncased__  
trainable params: 1,121,779 || all params: 68,458,982 || trainable%: 1.6386  
best training loss (20 epochs) = 0.060179  
best validation loss (20 epochs) = 0.153198  

__microsoft/deberta-v3-base__  
trainable params: 1,268,467 || all params: 186,074,342 || trainable%: 0.6817  _<-- with value layers_  
trainable params: 973,555 || all params: 185,779,430 || trainable%: 0.5240 _<-- without value layers_  
best training loss (20 epochs) = 0.238682  
best validation loss (20 epochs) = 0.303357  
achieved 0.267833 val loss on 5e-5 learning rate, then early stopping at epoch 15.

In [ ]:
# Multi Label Classification Fine Tuning
tagger = FineTuneLLM(
    model_name = MODEL_NAME,
    dataset = sf.dataset,
    n_labels = len(sf.unique_tags),
    label2id = sf.label2id,
    id2label = sf.id2label,
    class_weights = sf.class_weights
)

tagger.prepare_data(
    max_input_length = MAX_INPUT_LENGTH
)

tagger.train(
    batch_size = BATCH_SIZE,
    n_epochs = NUM_EPOCHS,
    learning_rate = LEARNING_RATE,
    weight_decay = WEIGHT_DECAY,
    # output_dir = f'{PROJECT_PATH}/models/scryfall_auto_tagger' # uncomment if in google colab
    output_dir = f'../models/scryfall_auto_tagger'
)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Loading weights:   0%|          | 0/198 [00:00<?, ?it/s]

DebertaV2ForSequenceClassification LOAD REPORT from: microsoft/deberta-v3-base
Key                                     | Status     | 
----------------------------------------+------------+-
lm_predictions.lm_head.dense.bias       | UNEXPECTED | 
mask_predictions.dense.weight           | UNEXPECTED | 
mask_predictions.classifier.weight      | UNEXPECTED | 
mask_predictions.classifier.bias        | UNEXPECTED | 
mask_predictions.LayerNorm.bias         | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.weight | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.bias   | UNEXPECTED | 
lm_predictions.lm_head.dense.weight     | UNEXPECTED | 
mask_predictions.LayerNorm.weight       | UNEXPECTED | 
lm_predictions.lm_head.bias             | UNEXPECTED | 
mask_predictions.dense.bias             | UNEXPECTED | 
pooler.dense.weight                     | MISSING    | 
pooler.dense.bias                       | MISSING    | 
classifier.weight                       | MISSING    | 
classifier.bias          

trainable params: 1,115,436 || all params: 185,768,280 || trainable%: 0.6004


Map:   0%|          | 0/6717 [00:00<?, ? examples/s]

...

Map:   0%|          | 0/1680 [00:00<?, ? examples/s]

Map:   0%|          | 0/50 [00:00<?, ? examples/s]

Using device: Tesla T4


/content/gdrive/MyDrive/data_science/scryfall-llm-sandbox/src/fine_tuning/fine_tune_multi_lab.py:32: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  self.pos_weight = torch.tensor(class_weights, dtype=torch.float32)


Epoch,Training Loss,Validation Loss,Micro Precision,Micro Recall,Micro F1,Macro Precision,Macro Recall,Macro F1
1,0.352980,0.359886,0.156958,0.180226,0.167789,0.013087,0.028455,0.013188
2,0.353308,0.345489,0.124369,0.189648,0.150223,0.023752,0.044014,0.018984


what if we try a lower tag count. Top 300 instead of Top 500.

In [ ]:
# SEQ2SEQ fine tune the model
if TASK == 'seq2seq':
    tagger = FineTuneLLM(
        model_name = MODEL_NAME,
        dataset = sf.dataset
    )
    tagger.prepare_data(
        max_input_length = MAX_INPUT_LENGTH,
        max_target_length = MAX_TARGET_LENGTH
    )
    tagger.train(
        batch_size = BATCH_SIZE,
        n_epochs = NUM_EPOCHS,
        learning_rate = LEARNING_RATE,
        weight_decay = WEIGHT_DECAY,
        generation_max_length = GENERATION_MAX_LENGTH,
        generation_num_beams = GENERATION_NUM_BEAMS,
        # output_dir = f'{PROJECT_PATH}/models/scryfall_auto_tagger' # uncomment if in google colab
        output_dir = f'../models/scryfall_auto_tagger'
    )

In [ ]:
if TASK == 'seq2seq':
    # Assuming 'tagger' is your FineTuneLLM instance
    from src.utils.debug_autotagger import debug_autotagger_outputs
    debug_autotagger_outputs(tagger, sf.dataset, num_samples=10)

In [ ]:
if TASK == 'multi_label_classification':
    import numpy as np
    test_ids = list(sf.dataset['test']['id'])
    sample_ids = np.random.choice(test_ids, 5, replace = False)
    for card in sf.dataset['test']:
        if card['id'] in sample_ids:
            pred = tagger.generate_tags(card_text = card['document'])
            print(f'Card ID {card["id"]}')
            print(f'\tActual Tags = {sorted(card["tags"])}')
            print(f'\tPredicted Tags = {sorted(pred)}')

In [ ]:
assert 1 == 0

## Save To Huggingface Hub

In [ ]:
# UNCOMMENT TO login to the hugging face
from huggingface_hub import notebook_login
# with open('../huggingface_token.txt', 'r') as f:
#     token = f.read()

notebook_login()

In [ ]:
# UNCOMMENT TO upload to huggingface hub
from huggingface_hub import get_full_repo_name
# from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
repo_id = get_full_repo_name(OUTPUT_DIR)
tagger.model.push_to_hub(repo_id)
tagger.tokenizer.push_to_hub(repo_id)

## Load Model From Huggingface

In [ ]:
# load model and use it to test results
from src.modeling.auto_tagger_multi_lab import ScryfallTaggerFromPretrained
tagger2 = ScryfallTaggerFromPretrained(
    base_model_name = MODEL_NAME,
    n_labels = len(sf.unique_tags),
    output_dir = OUTPUT_DIR,
    id2label = sf.id2label,
    label2id = sf.label2id
)

In [ ]:
# use the pretrained model on the test set
if TASK == 'multi_label_classification':
    import numpy as np
    test_ids = list(sf.dataset['test']['id'])
    sample_ids = np.random.choice(test_ids, 5, replace = False)
    for card in sf.dataset['test']:
        if card['id'] in sample_ids:
            pred = tagger2.generate_tags(card_text = card['document'], threshold = 0.4)
            print(f'Card ID {card["id"]}')
            print(f'\tActual Tags = {sorted(card["tags"])}')
            print(f'\tPredicted Tags = {sorted(pred)}')

## Graveyard

In [ ]:
# # upload the model to the huggingface hub
# from huggingface_hub import Repository
# from huggingface_hub import get_full_repo_name

# ## define the repo locally
# ## NOTE: Be sure to create OUTPUT_DIR in the hub manually first
# repo_name = get_full_repo_name(OUTPUT_DIR)
# repo = Repository(OUTPUT_DIR, clone_from = repo_name)

# ## save to hub
# tagger.save_to_huggingface_hub(
#     output_dir = OUTPUT_DIR,
#     repo = repo,
#     commit_message = f'Fine-tuned {MODEL_NAME} on scryfall tags.'
# )